# **Experiment Coding**
---
This tutorial is designed to guide you through the lab's standard operating procedure when coding experiment with the purpose of providing you with a clear, reproducible, and standardized way to build and run MEG experiments in our lab.<br>
You need to have [MATLAB](https://de.mathworks.com/products/matlab.html), [Psychtoolbox (PTB)](http://psychtoolbox.org/download.html), [GStreamer](https://gstreamer.freedesktop.org/download/#sources), and the [o_ptb](https://o-ptb.readthedocs.io/en/latest/install.html) installed.<br><br> For stimulus presentation, the lab uses the o_ptb, which is a class-based objective for Psychtoolbox written by [Thomas Hartmann](https://ccns.plus.ac.at/labs/auditory/members/thomas_hartmann/). You can find a tutorial on how to use the o_ptb [here](https://o-ptb.readthedocs.io/en/latest/tutorial.html) and a documentation of its functions [here](https://o-ptb.readthedocs.io/en/latest/reference.html).<br><br> If you code a new experiment, you can use the present tutorial to understand what the required structure is and what the functions do, and then clone the following repo (xxxxx) and adapt it to your needs (i.e., insert your experiment-specific code). For examples on SOP-conform experiment code, ask [Thomas Hartmann](https://ccns.plus.ac.at/labs/auditory/members/thomas_hartmann/) to add you to the GitLab folder [thht_experiments](https://gitlab.com/thht_experiments). <br><br>
Let's get started!<br><br><br>

This is the structure that all experiments follow: <br>

📁 +exp  
├── 📁 +init  
│   ├── 📄 cofig_ptb.m  
│   ├── 📄 init_ptb.m  
│   ├── 📄 prepare_cfg.m  
│   ├── 📄 prepare_stims.m  
│   └── 📄 prepare_subject.m  
├── 📁 +parts  
│   ├── 📄 auditory_threshold.m  
│   ├── 📄 instructions.m  
│   ├── 📄 resting.m  
│   └── 📄 tracking_block.m  
├── 📁 stims  
│   └── 📄 ExpStim.m <br>
📁 +functions  
├── 📄 FormattedText.m  
├── 📄 InstructionText.m  
├── 📄 ft_cfg2keyval.m  
├── 📄 ft_getopt.m  
└── 📄 iterproduct.m  
📁 external  
├── 📄 PsychometricFunction.m  
└── 📄 ThresholdHunter.m  
📁 instructions  
📁 soundfiles  
📄 run.m  

The central script is the `run.m` script (visible at the bottom of the tree above (i.e., 📄 run.m )). This is the master-script that is used in the lab to run the experiment. From there, all other scripts are accessed and executed. <br> 
- Note that it is central to successful data collection that people running the experiment can make as little mistakes as possible. Specifically, that means that the `run.m` script should be short and clear. It should call top‑level functions only.
- Keep it short and declarative. 
- Pass explicit parameters (e.g., `subject_id`, `block_nr`) and avoid hidden globals.
- Use the `exp.parts.*` methods to sequence blocks.

**Typical sequence:**
1) Load PTB configuration → 2) Init PTB → 3) Prepare cfg → 4) Prepare stim blocks → 5) Prepare subject → 6) Run parts

In [ ]:
% run.m
% master script to run the experiment

%% clean
clear all global
close all      
restoredefaultpath;

%% init
ptb_cfg = exp.init.config_ptb();

%% set variables
subject_id = 'test';

%% prepare subject
exp.init.prepare_subject(subject_id);

%% do resting state
exp.parts.resting(subject_id);

%% do auditory threshold
exp.parts.auditory_threshold(subject_id);

%% do instructions
exp.parts.instructions(subject_id);

%% do blocks
block_nr = 1; % change the number here for a different block
exp.parts.tracking_block(subject_id, block_nr); 

As you can see, it accesses a bunch of functions and further scripts. All of these can be found in the following. The scripts are ordered after the structure you can see in the tree above (i.e., config_ptb.m is the first script/function, soundfiles are described at the end of the tutorial). If you need any further custom functions, you can of course just insert them into your experiment wherever it makes sense. Just be aware of the nested code structure that the OOP style brings. <br><br><br>
Let's walk through the code! <br> <br>
📁 +exp  
├── 📁 +init  
│   ├── 📄 **cofig_ptb.m**

In [ ]:
% ptb_config: this is the configuration for the o_ptb

function ptb_config = config_ptb()
%% Tell matlab where gstreamer is stored and set up PTB
%setenv('DYLD_LIBRARY_PATH', ['xxx/gstreamer:', getenv('DYLD_LIBRARY_PATH')]);

%% init paths....
addpath('xxx/MATLAB/Toolboxes/o_ptb/');
o_ptb.init_ptb('xxx/MATLAB/Toolboxes/Psychtoolbox-3/');

%% create ptb_config
ptb_config = o_ptb.PTB_Config();
ptb_config.window_scale = 0.3;
ptb_config.flip_horizontal = false;
ptb_config.fullscreen = false;
ptb_config.skip_sync_test = true;
ptb_config.allow_wayland = true;
ptb_config.hide_mouse = true;
ptb_config.force_datapixx = false;
ptb_config.internal_config.blend_function = {'GL_SRC_ALPHA', 'GL_ONE_MINUS_SRC_ALPHA'};
ptb_config.internal_config.use_decorated_window = true;
% ptb_config.internal_config.trigger_subsystem = @th_ptb.subsystems.trigger.Dummy; %to make it run on debian
ptb_config.datapixxaudio_config.volume = [1 0.001];
%ptb_config.psychportaudio_config.device = 'pipewire';
ptb_config.psychportaudio_config.device = 'insert the name of your audio device here'; % e.g. 'Built-in Output' on Mac
ptb_config.psychportaudio_config.reqlatencyclass = 0;
ptb_config.psychportaudio_config.simulate_audio_cutoff = true;
ptb_config.audio_system_cutoff = 4000;

ptb_config.defaults.text_size = 70;
ptb_config.defaults.fixcross_size = 240;

%button config for bayesian staircase
ptb_config.keyboardresponse_config.button_mapping('yes') = KbName('LeftArrow');
ptb_config.keyboardresponse_config.button_mapping('no') = KbName('RightArrow');
ptb_config.datapixxresponse_config.button_mapping('yes') = ptb_config.datapixxresponse_config.Red;
ptb_config.datapixxresponse_config.button_mapping('no') = ptb_config.datapixxresponse_config.Green;

%button config for audiobook masking % add config for meg
ptb_config.keyboardresponse_config.button_mapping('next') = KbName('space');
ptb_config.datapixxresponse_config.button_mapping('next') = ptb_config.datapixxresponse_config.Yellow;
ptb_config.keyboardresponse_config.button_mapping('measure') = KbName('Return');

ptb_config.datapixxresponse_config.button_mapping('right') = ptb_config.datapixxresponse_config.Red;
ptb_config.keyboardresponse_config.button_mapping('right') = KbName('RightArrow');
ptb_config.datapixxresponse_config.button_mapping('left') = ptb_config.datapixxresponse_config.Green;
ptb_config.keyboardresponse_config.button_mapping('left') = KbName('LeftArrow');

ptb_config.datapixxresponse_config.button_mapping('false') = ptb_config.datapixxresponse_config.Red;
ptb_config.keyboardresponse_config.button_mapping('false') = KbName('RightArrow');
ptb_config.datapixxresponse_config.button_mapping('true') = ptb_config.datapixxresponse_config.Green;
ptb_config.keyboardresponse_config.button_mapping('true') = KbName('LeftArrow');

ptb_config.datapixxresponse_config.button_mapping('choose') = ptb_config.datapixxresponse_config.Yellow;
ptb_config.keyboardresponse_config.button_mapping('choose') = KbName('UpArrow');

% configure eyetracker
ptb_config.datapixxtrackpixx_config.lens = 1;
ptb_config.datapixxtrackpixx_config.distance = 110; %originally wrongly 82, now 110
ptb_config.datapixxtrackpixx_config.analogue_eye = 1; % (0 = left eye, 1 = right eye)

ptb_config.real_experiment_sbg_cdk(false); % set to true when running the real experiment in the MEG lab!
end


📁 +exp  
├── 📁 +init  
│   ├── 📄 cofig_ptb.m  
│   ├── 📄 **init_ptb.m** 

In [ ]:
function ptb = init_ptb(ptb_config)
%% init ptb...
ptb = o_ptb.PTB.get_instance(ptb_config);
ptb.setup_audio;
ptb.setup_trigger;
ptb.setup_screen();
ptb.setup_response;
ptb.setup_eyetracker;
end

📁 +exp  
├── 📁 +init  
│   ├── 📄 cofig_ptb.m  
│   ├── 📄 init_ptb.m  
│   ├── 📄 **prepare_cfg.m**

In [ ]:
function cfg = prepare_cfg()
%PREPARE_CFG Summary of this function goes here
%   Detailed explanation goes here
cfg = [];
cfg.data_path = 'subj_data';
cfg.eye_path = 'eye_data';

cfg.pre_trial_delay = 5;
cfg.extra_wait_time = 2;

%Path to all audiobook
cfg.sounds.basefolder = 'soundfiles';
cfg.sounds.speech = 'speech';
cfg.sounds.music = 'music';

% volume staircase
cfg.ml.db_range = -40:-0.5:-120;
cfg.ml.false_alarm = 0;
cfg.ml.slope = 0.5;
cfg.ml.start = -40;
cfg.ml.mean_onset_delay = 0.5;
cfg.ml.onset_jitter = 0.1;
cfg.ml.post_sound_delay = 2;
cfg.add_db = 40;
cfg.dev.lufs = -25;

% frequency shift
cfg.original.music.n_files = 16;
cfg.original.speech.n_files = 16;
cfg.original.shift_semitones = 4;
cfg.original.shift_n_seconds = 10;

%% settings for triggers
cfg.trigger.sounds.speech = 1;
cfg.trigger.sounds.music = 2;
cfg.trigger.sounds.music_target = 4;
cfg.trigger.correct = 8;
cfg.trigger.incorrect = 16;
end

📁 +exp  
├── 📁 +init  
│   ├── 📄 cofig_ptb.m  
│   ├── 📄 init_ptb.m  
│   ├── 📄 prepare_cfg.m  
│   ├── 📄 **prepare_stims.m**  

In [ ]:
function stim_blocks = prepare_stims()
% randomize music stimuli
stim_blocks = {};
stims_music = {};
stims_speech = {};

present_condition = {'name of condition 1', 'name of condition 2'};
idx = 1:16;
[p_cond, idx] = ndgrid(present_condition, idx);
all_p_cond = p_cond(:);
all_idx = idx(:);


for cur_idx = 1:16
    cur_present = all_p_cond{cur_idx};
    cur_file_idx = all_idx(cur_idx);

    if strcmp(cur_present, 'music')
        music_target = rand() > 0.5;
        cur_stim = exp.stims.ExpStim(false, cur_file_idx);
        cur_stim = cur_stim.set_music_target(music_target);
        stims_music{end+1} = cur_stim;
    elseif strcmp(cur_present, 'speech')
        cur_stim = exp.stims.ExpStim(true, cur_file_idx);
        stims_speech{end+1} = cur_stim;
    end
end %for


% shuffle
stims_music = Shuffle(stims_music);
stims_speech = Shuffle(stims_speech);

% sort into blocks
for idx_block = 1:3
    stim_blocks{idx_block} = Shuffle({stims_music{1:2}, stims_speech{1:2}});
    stims_music(1:2) = [];
    stims_speech(1:2) = [];
end %for
end

📁 +exp  
├── 📁 +init  
│   ├── 📄 cofig_ptb.m  
│   ├── 📄 init_ptb.m  
│   ├── 📄 prepare_cfg.m  
│   ├── 📄 prepare_stims.m  
│   └── 📄 **prepare_subject.m**

In [ ]:
function prepare_subject(subject_id) % not here that the subject_id is entered in the 'run.m' script by the experimenter in the lab
cfg = exp.init.prepare_cfg();

if ~exist(cfg.data_path)
    mkdir(cfg.data_path);
end %if

subject_fname = fullfile(cfg.data_path, sprintf('%s.mat', subject_id));
  
if exist(subject_fname)
    fprintf('The Subject is already prepared. You can go on\n');
    return;
end %if

stims = exp.init.prepare_stims();

subject_data = {};
subject_data.stims = stims;
subject_data.stims_struct = {};
warning('off', 'MATLAB:structOnObject');
for idx_block = 1:length(subject_data.stims)
    cur_block = subject_data.stims{idx_block};
    subject_data.stims_struct{end+1} = cellfun(@(x) struct(x), cur_block, UniformOutput=false);
end %for


subject_data.block_done = 0;
subject_data.blocks = {};

subject_data.resting_done = false;
subject_data.instructions_done = false;

save(subject_fname, 'subject_data');

end



📁 +exp  
├── 📁 +parts  
│   ├── 📄 auditory_threshold.m  
│   ├── 📄 instructions.m  
│   ├── 📄 resting.m  
│   └── 📄 **tracking_block.m**

In [ ]:
function auditory_threshold(subject_id)
%% get cfg
cfg = exp.init.prepare_cfg;

%% load subject_data
load(fullfile(cfg.data_path, subject_id));
if isfield(subject_data, 't_hunter')
  fprintf('\n\n\nYou already did the auditory threshold! Are you sure you want to run it again?\n');
  x = input('If you want to redo the auditory threshold, enter "YES". ', 's');
  
  if ~strcmp(x, 'YES')
    return
  end %if
end %if

%% init ptb
ptb_config = exp.init.config_ptb;
ptb = exp.init.init_ptb(ptb_config);

%% prepare...
stim_tmp = exp.stims.ExpStim(false, 1, false);
stim = stim_tmp.get_sound_object();
stim.cut(5.1, 7.1);
stim.fadeinout(0.02);
fixcross = o_ptb.stimuli.visual.FixationCross();

texts = [];
texts.question_text = o_ptb.stimuli.visual.Text('Haben Sie die Melodie gehört?');
texts.yes_text = o_ptb.stimuli.visual.Text('Ja: Roter Knopf');
texts.no_text = o_ptb.stimuli.visual.Text('Nein: Grüner Knopf');

texts.yes_text.sy = 700;
texts.no_text.sy = 770;

t_hunter = ml_threshold.ThresholdHunter(cfg.ml.start, cfg.ml.db_range, cfg.ml.false_alarm, cfg.ml.slope);

%% show instructions...
text = o_ptb.stimuli.visual.TextFile(fullfile('instructions', 'instructions_volume_staircase.txt'));
text.size = 50;
ptb.draw(text);
ptb.flip();

ptb.wait_for_keys('yes');

%% do it...
while ~t_hunter.stop
  ptb.draw(fixcross);
  now = ptb.flip();
  
  stim.db = t_hunter.current_probe_value;
  ptb.prepare_audio(stim);
  ptb.schedule_audio();
  ptb.play_on_flip();
  
  ptb.draw(fixcross);
  
  onset_delay = RandLim(1, cfg.ml.mean_onset_delay - cfg.ml.onset_jitter, cfg.ml.mean_onset_delay + cfg.ml.onset_jitter);
  
  now = ptb.flip(now + onset_delay);
  
  ptb.draw(texts.question_text);
  ptb.draw(texts.yes_text);
  ptb.draw(texts.no_text);
  
  ptb.flip(now + cfg.ml.post_sound_delay);
  
  resp = ptb.wait_for_keys({'yes', 'no'});
  
  t_hunter.process_response(strcmp(resp{1}, 'yes'));
end %while

%% check result...
sca;

if ~t_hunter.converged
  disp(t_hunter);
  error('The auditory threshold did not converge for some reason. Check the settings and retry.');
end %if

fprintf('Final db: %d\n', t_hunter.current_guess);

stim.db = t_hunter.current_guess + cfg.add_db;
subject_data.lufs = stim.lufs(1);


%% save
subject_data.t_hunter = t_hunter;
subject_data.threshold = t_hunter.current_guess;

save(fullfile(cfg.data_path, subject_id), 'subject_data');

fprintf('Auditory threshold done successfully.\n');

end

📁 +exp  
├── 📁 +parts  
│   ├── 📄 auditory_threshold.m  
│   ├── 📄 **instructions.m**  

In [ ]:
function instructions(subject_id)
%% init ptb
ptb_config = exp.init.config_ptb();
cfg = exp.init.prepare_cfg();

%% load subject_data
subject_fname = fullfile(cfg.data_path, sprintf('%s.mat', subject_id));
load(subject_fname);

load(fullfile(cfg.data_path, subject_id));
if ~isfield(subject_data, 't_hunter')
    fprintf('\n\n\nPlease do the audio_threshold first\n');
    return;
end %if

%% init the ptb
exp.init.init_ptb(ptb_config);
ptb = o_ptb.PTB.get_instance();

for idx_instruction = 1:3
    ptb.draw(functions.InstructionText(fullfile('instructions', sprintf('inst%02d.txt', idx_instruction))));
    ptb.flip();
    WaitSecs(1);
    ptb.wait_for_keys({'next'});
end %for

subject_data.instructions_done = true;
save(fullfile(cfg.data_path, subject_id), 'subject_data');
ptb.deinit();
end % function

📁 +exp  
├── 📁 +parts  
│   ├── 📄 auditory_threshold.m  
│   ├── 📄 instructions.m  
│   ├── 📄 **resting.m**

In [ ]:
function resting(subject_id)

%% get cfg
cfg = exp.init.prepare_cfg();

%% init ptb
ptb_config = exp.init.config_ptb();

%% load subject_data
subject_fname = fullfile(cfg.data_path, sprintf('%s.mat', subject_id));
load(subject_fname);


%% check whether resting has already been done
if subject_data.resting_done
  fprintf('\n\n\nWARNING!!! You already did the resting state run!! This is very probably NOT WHAT YOU WANT!\n');
  fprintf('If you really want to continue, press the "y" button now.\n\n\n\n');
  
  [~, key_pressed] = KbWait();
  if find(key_pressed) ~= KbName('y')
    return;
  end %if
end

%% init screen etc
ptb = exp.init.init_ptb(ptb_config);

%% verify eye positions & calibrate
calibration_folder = fullfile(cfg.eye_path,'calibration', subject_id);
if ~exist(calibration_folder,'dir')
    mkdir(calibration_folder)
end
calibration_file = fullfile(calibration_folder, 'resting.mat');
ptb.setup_eyetracker();
ptb.eyetracker_verify_eye_positions();
WaitSecs(1);
ptb.eyetracker_calibrate();
WaitSecs(1);
ptb.start_eyetracker();
WaitSecs(1);

%% Show instructions
HideCursor();
inst_text = o_ptb.stimuli.visual.Text('Einen Moment. Es geht gleich los...');
ptb.draw(inst_text);
ptb.flip();

KbWait();

%% show fixation cross...
fixcross = o_ptb.stimuli.visual.Text('+');
fixcross.size = 250;

ptb.draw(fixcross)
ptb.flip()

%% Wait for 5 minutes
WaitSecs(5*60+10);

%% save eye tracking data...
ptb.stop_eyetracker();
eye_folder = fullfile(cfg.eye_path,'eye_data', subject_id);
if ~exist(eye_folder,'dir')
    mkdir(eye_folder)
end
ptb.save_eyetracker_data(fullfile(eye_folder, 'resting.mat'));


%% Goodbye
ptb.deinit();
subject_data.resting_done = true;
save(fullfile(cfg.data_path, subject_id), 'subject_data');



📁 +exp  
├── 📁 +parts  
│   ├── 📄 auditory_threshold.m  
│   ├── 📄 instructions.m  
│   ├── 📄 resting.m  
│   └── 📄 **tracking_block.m**

In [ ]:
function tracking_block(subject_id, block_nr)
arguments
    subject_id string
    block_nr (1, 1) {mustBeInteger, mustBePositive, mustBeLessThanOrEqual(block_nr, 8)}
end %arguments

cfg = exp.init.prepare_cfg();

%% init ptb
ptb_config = exp.init.config_ptb();

%% load subject_data
subject_fname = fullfile(cfg.data_path, sprintf('%s.mat', subject_id));
load(subject_fname);

%% have we alread done resting?
if ~subject_data.instructions_done
    fprintf('Please show the instructions first.\n');
    return;
end %if

%% check whether we have already done this block
if block_nr <= subject_data.block_done
    fprintf('You have already done this block. Please do not do it again.\n');
    return;
end %if

%% check if we skip a block
if block_nr > subject_data.block_done + 1
    fprintf('You have not done the previous block. Please do that first.\n');
    return;
end %if

%% init the ptb
exp.init.init_ptb(ptb_config);
ptb = o_ptb.PTB.get_instance();

%% verify eye positions & calibrate
ptb.setup_eyetracker();
ptb.eyetracker_verify_eye_positions();
WaitSecs(1);
ptb.eyetracker_calibrate();
WaitSecs(1);
ptb.start_eyetracker();
WaitSecs(1);

%% do some more init
this_block = subject_data.stims{block_nr};
for idx = 1:length(this_block)
    this_block{idx}.get_sound_object();
end %for

%% show instructions
HideCursor();
inst_text = o_ptb.stimuli.visual.Text('Einen Moment. Es geht gleich los...');
ptb.draw(inst_text);
ptb.flip();

KbWait();
ptb.flip();

%% show fixcross
fixcross = o_ptb.stimuli.visual.FixationCross();
ptb.draw(fixcross);
ptb.flip();
this_block_data = {};


%% play stimuli....
Screen('Preference', 'TextRenderer', 1);

for idx_trial = 1:length(this_block)
    cur_trial_data = {};
    cur_trial = this_block{idx_trial};
    cur_sound = cur_trial.get_sound_object();
    cur_sound.lufs = subject_data.lufs;
    %cur_sound.lufs = -30;
    %if cfg.cut_for_dev
    %    cur_sound.cut(cur_sound.duration - cfg.cut_for_dev, cur_sound.duration);
    %end %if
    ptb.prepare_audio(cur_sound);
    ptb.prepare_trigger(cur_trial.trigger);
    ptb.prepare_trigger(cur_trial.file_idx, 0.1, true);
    
    ptb.draw(fixcross);
    flip_time = ptb.flip();

    ptb.schedule_audio();
    ptb.schedule_trigger();
    ptb.draw(fixcross);
    ptb.play_on_flip();

    flip_time = ptb.flip(flip_time + cfg.pre_trial_delay);


    if cur_trial.speech
        words = Shuffle({cur_trial.correct_last_noun; cur_trial.wrong_last_noun});
        question_text = functions.FormattedText(sprintf('Das letzte Hauptwort war:\n\n\n<color=ff0000>%s     <color=00ff00>%s', words{1}, words{2}));
        Screen('Preference', 'TextRenderer', 1);
        ptb.draw(question_text);
        ptb.flip(flip_time + cur_sound.duration + cfg.extra_wait_time);
        response = ptb.wait_for_keys({'right', 'left'});
        cur_trial_data.raw_response = response{1};
        cur_trial_data.words_to_choose = words;
        if strcmp(response{1}, 'left')
            chosen_word = words{1};
        else
            chosen_word = words{2};
        end %if

        correct = strcmp(chosen_word, cur_trial.correct_last_noun);
        cur_trial_data.correct = correct;
    else
        question_text = functions.FormattedText(sprintf('War die Musik am Ende höher?\n\n\n<color=ff0000>Ja     <color=00ff00>Nein'));
        Screen('Preference', 'TextRenderer', 1);
        ptb.draw(question_text);
        ptb.flip(flip_time + cur_sound.duration + cfg.extra_wait_time);
        response = ptb.wait_for_keys({'yes', 'no'});
        cur_trial_data.raw_response = response{1};
        bool_response = strcmp(response{1}, 'yes');
        correct = bool_response == cur_trial.music_target;
        cur_trial_data.correct = correct;
    end %if
    if correct
        ptb.prepare_trigger(cfg.trigger.correct);
    else
        ptb.prepare_trigger(cfg.trigger.incorrect);
    end
    ptb.schedule_trigger();
    ptb.play_on_flip();
    ptb.draw(fixcross);
    ptb.flip();


    this_block_data{idx_trial} = cur_trial_data;
end %for

%% save eye tracking data...
ptb.stop_eyetracker();
eye_folder = fullfile(cfg.eye_path,'eye_data', subject_id);
if ~exist(eye_folder,'dir')
    mkdir(eye_folder)
end
ptb.save_eyetracker_data(fullfile(eye_folder, ['/' sprintf('experiment_name_%d.mat', blk_n)]));



%% save
ptb.deinit();
subject_data.blocks{block_nr} = this_block_data;
subject_data.block_done = block_nr;

save(fullfile(cfg.data_path, subject_id), 'subject_data');


end



<br> The following is the script in which the stimuli are created. Adapt as needed. You can find an example from a previous experiment where vocoded music and speech was presented here.  <br>


📁 +exp  
├── 📁 stims  
│   └── 📄 **ExpStim.m**

In [ ]:
classdef ExpStim < handle & matlab.mixin.Copyable


    properties
        speech;
        file_idx;
        music_target;
        max_duration = 3*60;
        correct_last_noun = '';
        wrong_last_noun = '';
    end

    properties (Dependent)
        trigger;
        f_name_src;
        f_name_processed
    end

    properties (Access = protected, Transient)
        sound_object_cache
    end %properties

    methods (Static)
        function stims = get_all_stims()
            stims = {};
            total = 16;
            for cur_stimtype = {'music', 'speech'}
                for cur_idx = 1:total
                    if strcmp(cur_stimtype, 'speech')
                        stims{end+1} = exp.stims.ExpStim(true, cur_idx);
                    else
                        for music_target = {true, false}
                            music_target = music_target{1};
                            stims{end+1} = exp.stims.ExpStim(false, cur_idx, music_target);
                        end

                    end
                end
            
            end
        end
    end
    methods
        function obj = ExpStim(speech, file_idx, music_target)
            %VOCODEDSTIM Construct an instance of this class
            %   Detailed explanation goes here
            arguments
                speech logical
                file_idx (1, 1) {mustBeInteger, mustBePositive}
                music_target logical = false

            end

            obj.speech = speech;
            obj.file_idx = file_idx;
            obj.music_target = music_target;
          
            if speech
                warning('off', 'MATLAB:table:ModifiedAndSavedVarnames');
                noun_table = readtable('/Users/annika.etzler/Documents/MATLAB/Projects/Music and Speech Tracking/soundfiles/Speech/meta/last_nouns.csv', 'Delimiter', ',');
                warning('on', 'MATLAB:table:ModifiedAndSavedVarnames');
                [~, filename, ~] = fileparts(obj.f_name_src);
                row = noun_table(strcmp(noun_table.Filename, filename(1:3)), :);
                obj.correct_last_noun = row.LastNoun{1};
                obj.wrong_last_noun = row.WrongNoun{1};
            end %if
        end

        function obj = set_music_target(obj, music_target)
            obj.music_target = music_target;
        end

        function trigger = get.trigger(obj)
            cfg = exp.init.prepare_cfg();
            if obj.speech
                trigger =  cfg.trigger.sounds.speech;
            elseif ~obj.speech
                if obj.music_target
                    trigger =  cfg.trigger.sounds.music + cfg.trigger.sounds.music_target;
                else
                    trigger = cfg.trigger.sounds.music;
                end %if
            end
      
        end %function

        function f_name = get.f_name_src(obj)
            cfg = exp.init.prepare_cfg();
            if obj.speech
                f_path = fullfile(cfg.sounds.basefolder, cfg.sounds.speech, 'Original');
                all_wavs = dir(fullfile(f_path, '*.wav'));
                f_name = fullfile(f_path, all_wavs(obj.file_idx).name);
            else
                f_path = fullfile(cfg.sounds.basefolder, cfg.sounds.music, 'Original');
                all_mp3s = dir(fullfile(f_path, '*.mp3'));
                f_name = fullfile(f_path, all_mp3s(obj.file_idx).name);
            end %if
        end %function

        function f_name = get.f_name_processed(obj)
            if obj.speech
                error('f_name_processed should not be accessed for speech stimuli!');
            end
            if ~obj.music_target
                error('f_name_processed should not be accessed when music_target is false!');
            end
                cfg = exp.init.prepare_cfg();
                f_path = fullfile(cfg.sounds.basefolder, cfg.sounds.music, 'Repitched');
                f_name = fullfile(f_path, sprintf('music_target_%d_%02d_repitched.wav', obj.music_target, obj.file_idx));
        end

      function out = get_sound_object(obj)
            if isempty(obj.sound_object_cache)
                if obj.speech
                    % Load speech sound
                    obj.sound_object_cache = o_ptb.stimuli.auditory.Wav(obj.f_name_src);
                else
                    % For music
                    if obj.music_target
                        % Check if repitched file already exists
                        if isfile(obj.f_name_processed)
                            fprintf('Loading existing repitched file...\n');
                            obj.sound_object_cache = o_ptb.stimuli.auditory.Wav(obj.f_name_processed);
                        else
                            % Generate and save repitched file if it doesn't exist
                            fprintf('Repitched file not found. Generating repitched file...\n');
                            obj.sound_object_cache = obj.generate_sound_object();
                            obj.sound_object_cache.save_wav(obj.f_name_processed);
                            fprintf('Repitched file saved: %s\n', obj.f_name_processed);
                        end
                    else
                        % Load original music file
                        obj.sound_object_cache = o_ptb.stimuli.auditory.Wav(obj.f_name_src);
                    end
                end
            end
            out = obj.sound_object_cache;
        end
    end
      

    methods (Access = public)
        function out_sound = generate_sound_object(obj)
            cfg = exp.init.prepare_cfg();

            fprintf('Loading...\n');
            out_sound = o_ptb.stimuli.auditory.Wav(obj.f_name_src);
            fprintf('Cropping...\n');
            out_sound.crop_silence();
            if ~obj.speech
                out_sound.cut(0, obj.max_duration);
            end %if

            if obj.music_target
                fprintf('Repitching...\n');                
                snd_data = out_sound.get_sound_data(out_sound.s_rate, 2);
            
                first_pitch_sample = cfg.original.shift_n_seconds * out_sound.s_rate;
                snd_data_to_pitch = snd_data(end - first_pitch_sample:end, :);
                snd_data_pitched = shiftPitch(snd_data_to_pitch, cfg.original.shift_semitones);
                snd_data(end - first_pitch_sample:end, :) = snd_data_pitched;
                                               
                out_sound = o_ptb.stimuli.auditory.FromMatrix(snd_data', out_sound.s_rate);
            
                out_sound.db = -20; 
            else
                out_sound.db = -1; 

            end
            

        end %function
    end %protected methods
end

---
📁 +functions  
├── 📄 **FormattedText.m**

In [ ]:
classdef FormattedText < o_ptb.stimuli.visual.Text
    methods
        function on_draw(obj, ptb)
            obj.check_all_properties_set();
            
            old_size = ptb.screen('TextSize', obj.size);
            old_style = ptb.screen('TextStyle', obj.style);
            old_font = ptb.screen('TextFont', obj.font);
            
            DrawFormattedText2(obj.text, 'win', ptb.win_handle, 'sx', obj.sx, 'sy', obj.sy, 'wrapat', obj.wrapat, 'vSpacing', obj.vspacing, 'xalign', 'center', 'xlayout', 'center');
            
            ptb.screen('TextSize', old_size);
            ptb.screen('TextStyle', old_style);
            ptb.screen('TextFont', old_font);
            
        end
    end
end

📁 +functions  
├── 📄 FormattedText.m  
├── 📄 **InstructionText.m**

In [ ]:
classdef InstructionText < o_ptb.stimuli.visual.TextFile
    properties
    end
    
    methods
        function obj = InstructionText(filename)
            obj = obj@o_ptb.stimuli.visual.TextFile(filename);
            obj.wrapat = 75;
            obj.size = 46;
        end
    end
end

📁 +functions  
├── 📄 FormattedText.m  
├── 📄 InstructionText.m  
├── 📄 **ft_cfg2keyval.m**

In [ ]:
function [optarg] = ft_cfg2keyval(cfg)

% FT_CFG2KEYVAL converts between a structure and a cell-array with key-value
% pairs which can be used for optional input arguments.
%
% Use as
%   optarg = ft_cfg2keyval(cfg)
%
% See also FT_KEYVAL2CFG, FT_GETOPT

% Copyright (C) 2006-2016, Robert Oostenveld
%
% This file is part of FieldTrip, see http://www.fieldtriptoolbox.org
% for the documentation and details.
%
%    FieldTrip is free software: you can redistribute it and/or modify
%    it under the terms of the GNU General Public License as published by
%    the Free Software Foundation, either version 3 of the License, or
%    (at your option) any later version.
%
%    FieldTrip is distributed in the hope that it will be useful,
%    but WITHOUT ANY WARRANTY; without even the implied warranty of
%    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
%    GNU General Public License for more details.
%
%    You should have received a copy of the GNU General Public License
%    along with FieldTrip. If not, see <http://www.gnu.org/licenses/>.
%
% $Id$

if ~isempty(cfg)
  optarg = [fieldnames(cfg) struct2cell(cfg)]';
  optarg = optarg(:)';
else
  optarg = {};
end

📁 +functions  
├── 📄 FormattedText.m  
├── 📄 InstructionText.m  
├── 📄 ft_cfg2keyval.m  
├── 📄 **ft_getopt.m**

In [ ]:
function val = ft_getopt(opt, key, default, emptymeaningful)

% FT_GETOPT gets the value of a specified option from a configuration structure
% or from a cell-array with key-value pairs.
%
% Use as
%   val = ft_getopt(s, key, default, emptymeaningful)
% where the input values are
%   s               = structure or cell-array
%   key             = string
%   default         = any valid MATLAB data type
%   emptymeaningful = boolean value (optional, default = 0)
%
% If the key is present as field in the structure, or as key-value
% pair in the cell-array, the corresponding value will be returned.
%
% If the key is not present, ft_getopt will return an empty array.
%
% If the key is present but has an empty value, then the emptymeaningful
% flag specifies whether the empty value or the default value should
% be returned. If emptymeaningful==true, then an empty array will be
% returned. If emptymeaningful==false, then the specified default will
% be returned.
%
% See also FT_SETOPT, FT_CHECKOPT

% Copyright (C) 2011-2012, Robert Oostenveld
%
% This file is part of FieldTrip, see http://www.fieldtriptoolbox.org
% for the documentation and details.
%
%    FieldTrip is free software: you can redistribute it and/or modify
%    it under the terms of the GNU General Public License as published by
%    the Free Software Foundation, either version 3 of the License, or
%    (at your option) any later version.
%
%    FieldTrip is distributed in the hope that it will be useful,
%    but WITHOUT ANY WARRANTY; without even the implied warranty of
%    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
%    GNU General Public License for more details.
%
%    You should have received a copy of the GNU General Public License
%    along with FieldTrip. If not, see <http://www.gnu.org/licenses/>.
%
% $Id$

if nargin<3
  default = [];
end

if nargin < 4
  emptymeaningful = 0;
end

if isa(opt, 'struct') || isa(opt, 'config')
  % get the key-value from the structure
  fn = fieldnames(opt);
  if ~any(strcmp(key, fn))
    val = default;
  else
    val = opt.(key);
  end

elseif isa(opt, 'cell')
  % get the key-value from the cell-array
  if mod(length(opt),2)
    error('optional input arguments should come in key-value pairs, i.e. there should be an even number');
  end

  % the 1st, 3rd, etc. contain the keys, the 2nd, 4th, etc. contain the values
  keys = opt(1:2:end);
  vals = opt(2:2:end);

  % the following may be faster than cellfun(@ischar, keys)
  valid = false(size(keys));
  for i=1:numel(keys)
    valid(i) = ischar(keys{i});
  end

  if ~all(valid)
    error('optional input arguments should come in key-value pairs, the optional input argument %d is invalid (should be a string)', i);
  end

  hit = find(strcmpi(key, keys));
  if isempty(hit)
    % the requested key was not found
    val = default;
  elseif length(hit)==1
    % the requested key was found
    val = vals{hit};
  else
    error('multiple input arguments with the same name');
  end

elseif isempty(opt)
  % no options are specified, return default
  val = default;
end % isstruct or iscell or isempty

if isempty(val) && ~isempty(default) && ~emptymeaningful
  % use the default value instead of the empty input that was specified:
  % this applies for example if you do functionname('key', []), where
  % the empty is meant to indicate that the user does not know or care
  % what the value is
  val = default;
end


📁 +functions  
├── 📄 FormattedText.m  
├── 📄 InstructionText.m  
├── 📄 ft_cfg2keyval.m  
├── 📄 ft_getopt.m  
└── 📄 **iterproduct.m**

In [ ]:
function [ args ] = iterproduct(varargin)
% CIMEC_CELLARGS Returns a cell array with all possible combinations of the
% input parameters provided.
%
% You can provide any number of input arguments. If it is a cell array,
% every entry of it will be combined with every entry of all other cells.
%
% Example:
% args = obob_cellargs({1, 2, 3}, {5, 6})
% args{:}
% ans = 
%     [1]    [5]
% ans = 
%     [2]    [5]
% ans = 
%     [3]    [5]
% ans = 
%     [1]    [6]
% ans = 
%     [2]    [6]
% ans = 
%     [3]    [6]

% Copyright (c) 2014-2016, Thomas Hartmann
%
% This file is part of the obob_ownft distribution, see: https://gitlab.com/obob/obob_ownft/
%
%    obob_ownft is free software: you can redistribute it and/or modify
%    it under the terms of the GNU General Public License as published by
%    the Free Software Foundation, either version 3 of the License, or
%    (at your option) any later version.
%
%    obob_ownft is distributed in the hope that it will be useful,
%    but WITHOUT ANY WARRANTY; without even the implied warranty of
%    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
%    GNU General Public License for more details.
%
%    You should have received a copy of the GNU General Public License
%    along with obob_ownft. If not, see <http://www.gnu.org/licenses/>.
%
%    Please be aware that we can only offer support to people inside the
%    department of psychophysiology of the university of Salzburg and
%    associates.

nargs = length(varargin);

% checks whether all variable input arguments are wrapped in cells. If not,
% put it in a cell.
% Also, this loop creates the cell array 'alllengths' which holds an
% enumeration of every argument. This is useful for later combination.

alllengths = cell(1, nargs);
for i = 1:nargs
  if ~iscell(varargin{i})
    varargin{i} = {varargin{i}};
  end %if
  alllengths{i} = 1:length(varargin{i});
end %for

% use ndgrid to create all possible combinations.
[allgridcell{1:nargs}] = ndgrid(alllengths{:});
for i = 1:length(allgridcell)
  allgridcell{i} = allgridcell{i}(:);
end %for

for i = 1:length(allgridcell{1})
  args{i} = cell(1, nargs);
  for j = 1:nargs
    args{i}{j} = varargin{j}{allgridcell{j}(i)};
  end %for
end %for
end

---
📁 external  
├── 📄 **PsychometricFunction.m**

In [ ]:
classdef PsychometricFunction < handle
  %PSYCHOMETRICFUNCTION Summary of this class goes here
  %   Detailed explanation goes here
  
  properties(SetAccess=protected, GetAccess=public)
    mean;
    false_alarm;
    slope;
    p;
  end
  
  properties (Dependent)
    sweetpoint;
    sweetpoint_value;
  end %dependent properties
  
  methods
    function obj = PsychometricFunction(mean, false_alarm, slope)
      obj.mean = mean;
      obj.false_alarm = false_alarm;
      obj.slope = slope;
      obj.p = 1;
    end
    
    function val = get_value_from_p(obj, p)
      val = (log((obj.false_alarm - p) ./ (p - 1)) + obj.slope * obj.mean) / obj.slope;
    end %function
    
    function p = get_p_from_value(obj, val)
      p = obj.false_alarm + (1 - obj.false_alarm) * (1 ./ (1 + exp(-1 * obj.slope * (val - obj.mean))));
    end %function
    
    function sweetpoint = get.sweetpoint(obj)
      sweetpoint = (2 * obj.false_alarm + 1 + sqrt(1 + 8 * obj.false_alarm)) / (3 + sqrt(1 + 8 * obj.false_alarm));
    end %function
    
    function sweetpoint_val = get.sweetpoint_value(obj)
      sweetpoint_val = obj.get_value_from_p(obj.sweetpoint);
    end %function
    
    function process_response(obj, response, value)
      x = obj.get_p_from_value(value);
      if ~response
        x = 1 - x;
      end %if
      obj.p = obj.p * x;
    end %function
  end
end

📁 external  
├── 📄 PsychometricFunction.m  
└── 📄 **ThresholdHunter.m** 

In [ ]:
classdef ThresholdHunter < handle
  properties(SetAccess=protected, GetAccess=public)
    psych_functions = {};
    start_values;
    values_range;
    false_alarm_range;
    slopes_range;
    min_trials;
    max_trials;
    use_n_trials_for_stop;
    n_trials;
    guesses;
    responses;
    stimulated_values;
  end
  
  properties(Dependent)
    best_function;
    current_probe_value;
    current_guess;
    std_last_trials;
    stop_std;
    converged;
    stop;
    weighed_ps;
    raw_ps;
  end %dependend properties
  
  methods
    function obj = ThresholdHunter(start_values, values_range, false_alarm_range, slopes_range, min_trials, max_trials, use_n_trials_for_stop)
      if nargin < 5
        min_trials = 12;
      end %if
      
      if nargin < 6
        max_trials = 40;
      end %if
      
      if nargin < 7
        use_n_trials_for_stop = 6;
      end %if
      
      obj.start_values = start_values;
      obj.values_range = values_range;
      obj.false_alarm_range = false_alarm_range;
      obj.slopes_range = slopes_range;
      obj.min_trials = min_trials;
      obj.max_trials = max_trials;
      obj.use_n_trials_for_stop = use_n_trials_for_stop;
      obj.n_trials = 0;
      obj.guesses = [];
      obj.responses = [];
      obj.stimulated_values = [];
      
      for cur_val = obj.values_range(:)'
        for cur_fa = obj.false_alarm_range(:)'
          for cur_slope = obj.slopes_range(:)'
            obj.psych_functions{end+1} = ml_threshold.PsychometricFunction(cur_val, cur_fa, cur_slope);
          end %for
        end %for
      end %for
    end %function
    
    function best_f = get.best_function(obj)
      ps = cellfun(@(x) x.p, obj.psych_functions);
      [~, sortIdx] = sort(ps, 'descend');
      
      f_sorted = obj.psych_functions(sortIdx);
      
      best_f = f_sorted{1};
    end %function
    
    function current_probe_value = get.current_probe_value(obj)
      if (obj.n_trials + 1) <= length(obj.start_values)
        current_probe_value = obj.start_values(obj.n_trials + 1);
      else
        current_probe_value = obj.best_function.sweetpoint_value;
      end %if
    end %function
    
    function current_guess = get.current_guess(obj)
      current_guess = obj.best_function.mean;
    end %function
    
    function std_last_trials = get.std_last_trials(obj)
      if length(obj.guesses) < obj.use_n_trials_for_stop
        std_last_trials = nan;
      else
        std_last_trials = std(obj.guesses(end-obj.use_n_trials_for_stop+1:end));
      end %if
    end %function
    
    function stop_std = get.stop_std(obj)
      stop_std = abs(max(diff(obj.values_range)));
    end %function
    
    function converged = get.converged(obj)
      converged = obj.std_last_trials < obj.stop_std && obj.n_trials > length(obj.start_values);
    end %function
    
    function stop = get.stop(obj)
      stop = (obj.n_trials >= (obj.min_trials + length(obj.start_values)) & obj.converged) | obj.n_trials >= obj.max_trials;
    end %function
    
    function p = get.raw_ps(obj)
      p = cellfun(@(x) x.p, obj.psych_functions);
    end %function
    
    function wp = get.weighed_ps(obj)
      ps = cellfun(@(x) x.p, obj.psych_functions);
      wp = ps ./ sum(ps);
    end %function
    
    function process_response(obj, response)
      cur_value = obj.current_probe_value;
      obj.n_trials = obj.n_trials + 1;      
      
      cellfun(@(x) x.process_response(response, cur_value), obj.psych_functions);
      
      obj.guesses(end+1) = obj.current_guess;
      obj.responses(end+1) = response;
      obj.stimulated_values(end+1) = cur_value;
    end %function
    
    function str = to_string(obj)
      str = sprintf('ThresholdHunter. Trial %d/%d\nCurrent guess: %d Current std: %d\n', obj.n_trials, obj.max_trials, obj.current_guess, obj.std_last_trials);
    end %for
  end
end



---
📁 instructions <br> These should be adapted to the experiment-specific instructions where needed. Note that these are `.txt` files and not `.m` files. For illustrational purposes, you can see four `.txt` files from a recent experiment on music and speech tracking below. Adapt as needed.

*% inst01.txt* <br> <br>
*Guten Tag und vielen Dank, dass Sie mitmachen!* <br>

*Bitte lesen Sie sich diese Instruktionen genau durch.* <br>

*Drücken Sie bitte immer den roten Knopf zum weitermachen.*

*% inst02.txt*<br> <br>
*Sehr gut.* <br>

*Bei diesem Experiment hören Sie kurze Musikstücke und Ausschnitte aus einem Hörbuch. Manchmal wird das ganz normal klingen, manchmal jedoch verfremdet.* <br> <br>

*Sie haben immer eine von zwei Aufgaben:*<br> <br>

*1. Wenn Sie ein Hörbuch hören, konzentrieren Sie sich bitte auf den Text. Am Ende fragen wir Sie, welches das letzte Hauptwort war.* <br>
*2. Wenn Sie ein Musikstück hören, konzentrieren Sie sich bitte darauf. Es kann sein, dass alles am Ende etwas höher klingt. Danach werden Sie gefragt.* <br><br>

*Drücken Sie bitte den roten Knopf zum weitermachen.*

*% inst03.txt*<br> <br>*Ansonsten ist es wichtig, dass Sie die ganze Zeit auf das Kreuz in der Mitte des Bildschirms schauen.*<br><br>

*Wenn Sie Fragen haben, können Sie sich gerne jederzeit an die Versuchsleitung wenden.*<br><br>

*Drücken Sie bitte den roten Knopf zum weitermachen.*

*% instructions_volume_staircase.txt*<br> <br>

*Jetzt wollen wir feststellen, ab welcher Intensität sie etwas hören.*<br> <br>

*Es werden Ihnen nun Töne unterschiedlicher Lautstärke präsentiert.*<br> <br>

*Sie werden dann gefragt, ob Sie etwas gehört haben oder nicht.*<br>
*Bitte antworten Sie mit dem entsprechenden Knopf.*<br> <br>

*Bitte drücken Sie auf den roten Knopf um anzufangen.*

---
📁 soundfiles <br> <br>
Load the soundfiles you need in here.

---
<br> <br> Congrats, this is the end! If you have further questions, don't hesitate to ask lab members.